<h1 style='font-weight:bold; color:#003366; text-align:center;'>Tema 3: Funciones</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

## Objetivos del notebook

1. Reconocer las partes de una función y por qué `return` no es lo mismo que `print`.
2. Elegir bien entre argumentos posicionales, con nombre y valores por omisión.
3. Partir una función que hace cuatro cosas en cuatro funciones que hacen una cada una.
4. Validar lo que entra: `raise` para los datos absurdos, `try/except` para lo que sabés que
   va a pasar.
5. Construir un módulo de funciones aplicado a un caso real de finanzas personales.

<h2 style='font-weight:bold; color:#003366;'>Datos: Estado de cuenta</h2>

Los datos de hoy son una **lista de diccionarios**, donde cada diccionario es una fila y cada
clave es una columna. Es exactamente la forma en la que llega un estado de cuenta bancario, un archivo
CSV o la respuesta de una API.

In [9]:
# Cada diccionario es una transacción. La lista es el extracto del mes.
transacciones_de_ejemplo = [
    {"fecha": "2026-03-02", "descripcion": "Salario mensual", "monto": 52_000},
    {"fecha": "2026-03-03", "descripcion": "Alquiler apartamento", "monto": -14_500},
    {"fecha": "2026-03-07", "descripcion": "Tienda Inglesa", "monto": -6_240},
    {"fecha": "2026-03-11", "descripcion": "Cuota universidad", "monto": -9_800},
    {"fecha": "2026-03-15", "descripcion": "Netflix", "monto": -390},
    {"fecha": "2026-03-22", "descripcion": "Freelance", "monto": 8_000},
]

# Acceder a una clave de un diccionario suelto
print("Mirar un unico ejemplo de transacción:")
primera = transacciones_de_ejemplo[0]
print("Descripción:", primera["descripcion"])
print("Monto: $", primera["monto"])
print()

# Recorrer la lista: en cada vuelta, la variable es un diccionario entero
balance = 0
for transaccion in transacciones_de_ejemplo:

    signo = "ingreso" if transaccion["monto"] > 0 else "egreso "
    print(f"{transaccion['fecha']}  {signo}  {transaccion['descripcion']:<22} "
          f"$ {transaccion['monto']:>9,.0f}")
    
    balance = balance + transaccion["monto"]

print("\nBalance mensual: $U {:,.0f}".format(balance))


Mirar un unico ejemplo de transacción:
Descripción: Salario mensual
Monto: $ 52000

2026-03-02  ingreso  Salario mensual        $    52,000
2026-03-03  egreso   Alquiler apartamento   $   -14,500
2026-03-07  egreso   Tienda Inglesa         $    -6,240
2026-03-11  egreso   Cuota universidad      $    -9,800
2026-03-15  egreso   Netflix                $      -390
2026-03-22  ingreso  Freelance              $     8,000

Balance mensual: $U 29,070


<h2 style='font-weight:bold; color:#003366;'>1. Importancia de las funciones</h2>

1. **Se puede revisar por separado.** Ya lo dijimos, y es la más importante.
2. **Se puede reutilizar.** La función que calcula la tasa de ahorro del mes sirve igual para el
   trimestre y para el año.
3. **Se puede corregir sin miedo.** Si cambia el criterio de clasificación, se toca una función
   y no hay que buscar dónde más aparecía ese `if`.

<h2 style='font-weight:bold; color:#003366;'>2. Anatomía de una función</h2>

Repasemos las partes con nombre, porque las vamos a usar todo el semestre.

Vocabulario que conviene tener claro, porque son tres cosas distintas que se confunden todo el
tiempo:

- **Parámetro**: el nombre que aparece en la definición (`capital`).
- **Argumento**: el valor concreto que se pasa al llamarla (`120_000`).
- **Firma**: el nombre de la función más sus parámetros. Es el contrato público: es lo único que
  alguien necesita leer para usarla.

In [12]:
# Anatomía, parte por parte:
#
#   def calcular_interes(capital, tasa=0.05):
#   ^^^ ^                ^^^^^^^  ^^^^^^^^^
#    |  |                   |         |
#    |  |                   |     valor por defecto: si no se pasa, se usa este.
#    |  |                parámetros
#    | nombre de la función
#   palabra clave que abre la definición
#
#       """Docstring: una línea que dice qué hace."""
#       return capital * tasa
#       ^^^^^^ lo que la función DEVUELVE a quien la llamó

def calcular_interes(capital, tasa=0.05):
    """Interés simple de un período sobre el capital indicado."""
    return capital * tasa


print(f"Con la tasa por defecto (5%): $U {calcular_interes(120_000):,.2f}")
print(f"Con una tasa de 9%: $U {calcular_interes(120_000, tasa=0.09):,.2f}")

Con la tasa por defecto (5%): $U 6,000.00
Con una tasa de 9%: $U 10,800.00


<h2 style='font-weight:bold; color:#003366;'>3. `return` contra `print`: la distinción más importante del tema</h2>

- Una función que **imprime** sirve para *mirar* un número en pantalla. Y nada más.
- Una función que **devuelve** sirve para graficarlo, exportarlo a Excel, meterlo en una tabla,
  compararlo con otro, o pasarlo a otra función.

Cuando una función imprime en lugar de devolver, el número existe durante medio segundo en la
salida de la celda y después se pierde. No hay forma de sumarlo, ordenarlo ni graficarlo.

Y hay un detalle que sorprende: una función sin `return` **no devuelve nada**. Técnicamente
devuelve `None`, que es el valor de Python para "nada". Mirá lo que pasa cuando se intenta
guardar el resultado de una función que solo imprime.

In [14]:
# Versión que solo imprime
def mostrar_cuota(monto, tasa_anual, plazo_meses):
    """Imprime la cuota en pantalla. No devuelve nada."""
    tasa_mensual = tasa_anual / 12
    factor = (1 + tasa_mensual) ** plazo_meses
    print(f"Cuota: $U {monto * tasa_mensual * factor / (factor - 1):,.2f}")


# Versión que devuelve
def calcular_cuota(monto, tasa_anual, plazo_meses):
    """Cuota fija mensual de un préstamo con sistema francés.

    Si la tasa es cero la cuota es el monto dividido el plazo. Hay que tratarlo aparte
    para no dividir por cero.
    """
    tasa_mensual = tasa_anual / 12
    if tasa_mensual == 0:
        return monto / plazo_meses

    factor = (1 + tasa_mensual) ** plazo_meses
    return monto * tasa_mensual * factor / (factor - 1)


resultado = mostrar_cuota(300_000, 0.24, 36)
print("Lo que devolvió mostrar_cuota:", resultado, "(no se puede hacer nada con esto)")
print()

# Con la que devuelve se puede armar una tabla, ordenarla, graficarla, exportarla
plazos = [12, 24, 36, 48, 60]
print("Préstamo de $U 300.000 al 24% anual\n")
for plazo in plazos:
    cuota = calcular_cuota(monto=300_000, tasa_anual=0.24, plazo_meses=plazo)
    total_pagado = cuota * plazo
    print(f"{plazo:>3} meses -> cuota $U {cuota:>8,.0f} | "
          f"total pagado $U {total_pagado:>9,.0f} | "
          f"intereses $U {total_pagado - 300_000:>8,.0f}")

Cuota: $U 11,769.86
Lo que devolvió mostrar_cuota: None (no se puede hacer nada con esto)

Préstamo de $U 300.000 al 24% anual

 12 meses -> cuota $U   28,368 | total pagado $U   340,415 | intereses $U   40,415
 24 meses -> cuota $U   15,861 | total pagado $U   380,672 | intereses $U   80,672
 36 meses -> cuota $U   11,770 | total pagado $U   423,715 | intereses $U  123,715
 48 meses -> cuota $U    9,781 | total pagado $U   469,466 | intereses $U  169,466
 60 meses -> cuota $U    8,630 | total pagado $U   517,823 | intereses $U  217,823


**Interpretación** Estirar el plazo de 12 a 60 meses baja la cuota de
$U 28.368 a $U 8.630, un 70% menos, y ese es el argumento de venta. Pero los intereses pasan de
$U 40.415 a $U 217.823: **más de cinco veces**. La misma deuda cuesta $U 177.000 más por el
solo hecho de pagarla en cinco años en vez de en uno.

Ese cuadro es el resultado de haber usado `return`. Con `mostrar_cuota` no existía, porque no
había manera de juntar los cinco números en una tabla.

<h2 style='font-weight:bold; color:#003366;'>4. Argumentos: posicionales, con nombre y valores por omisión</h2>

Python admite pasar los argumentos de dos maneras, y la elección tiene consecuencias reales.

- **Posicionales**: se asignan por orden. Cómodos cuando hay uno o dos parámetros obvios.
- **Con nombre**: se asignan por nombre, en cualquier orden. Imprescindibles cuando hay varios
  parámetros del mismo tipo, porque eliminan la posibilidad de confundirlos.

> **Criterio del curso:** si una función tiene **tres parámetros o más, llamala con nombres.**
> Es una de las cosas que más se valora al corregir los prácticos, y la razón está en la celda
> que sigue.

In [15]:
# Las dos formas dan el mismo resultado
print(f"Posicional  : $U {calcular_cuota(300_000, 0.24, 36):,.2f}")
print(f"Con nombres : $U {calcular_cuota(monto=300_000, plazo_meses=36, tasa_anual=0.24):,.2f}")
print()

# El caso peligroso: se invirtieron la tasa y el plazo. Python NO protesta,
# porque los tres argumentos son números y el orden es formalmente válido.
print(f"Orden mezclado: $U {calcular_cuota(300_000, 36, 0.24):,.2f}")

Posicional  : $U 11,769.86
Con nombres : $U 11,769.86

Orden mezclado: $U 3,179,960.58


Una cuota de **$U 3.179.960** para un préstamo de $U 300.000. Python no dio ningún
error: interpretó una tasa anual del 3.600% y un plazo de 0,24 meses, hizo la cuenta y devolvió
un número.

Este es el bug más peligroso que existe en análisis de datos, porque **no falla**. Se imprime en
un informe, alguien lo lee, y nadie se da cuenta hasta que un cliente pregunta. Llamar con
nombres hace que sea imposible.

<h2 style='font-weight:bold; color:#003366;'>5. Una función, una responsabilidad</h2>

En el Tema 1 vimos el principio: una función tiene que poder describirse en una frase sin usar la
palabra "y". Acá lo aplicamos al código que de verdad vamos a escribir.

Mirá esta función, que es la que aparece naturalmente cuando uno se sienta a resolver el
problema:

```python
def analizar(ruta):
    transacciones = []
    with open(ruta, encoding="utf-8") as archivo:
        for fila in csv.DictReader(archivo):
            try:
                fila["monto"] = float(fila["monto"])
            except ValueError:
                continue
            if not fila["categoria"]:
                if "netflix" in fila["descripcion"].lower():
                    fila["categoria"] = "Ocio"
                elif "alquiler" in fila["descripcion"].lower():
                    fila["categoria"] = "Vivienda"
            transacciones.append(fila)
    total = sum(t["monto"] for t in transacciones if t["monto"] < 0)
    print("Gasto total:", total)
```

Se describe así: **carga** el archivo **y** descarta filas malas **y** imputa categorías **y**
suma **e** imprime. Cinco "y". Son cinco funciones.

Y el problema no es estético. Con esa función no podés:

- saber **cuántas filas** se descartaron, porque el `continue` las tira en silencio,
- probar la imputación de categorías sin volver a leer el archivo del disco,
- calcular los ingresos, porque solo suma los negativos,
- hacer nada con el total, porque se imprime y se pierde.

En la sección 7 vamos a escribir la versión partida, de a una función por vez, con estos nombres:

| Función | Responsabilidad |
|---|---|
| `cargar_transacciones` | Leer el archivo y convertir el monto. Informar cuántas filas descartó |
| `clasificar_por_palabra_clave` | Dada una descripción, decir qué categoría le corresponde |
| `completar_categorias` | Rellenar las categorías vacías. Informar cuántas quedaron sin clasificar |
| `separar_ingresos_y_egresos` | Partir la lista en dos |
| `sumar_montos` | Sumar una lista de transacciones |
| `calcular_tasa_ahorro` | Excedente sobre ingresos |
| `resumir_por_mes` | Ingresos, egresos y balance de cada mes |

Fijate que ninguna imprime. Todas devuelven. Los `print` van a estar afuera, una sola vez, al
final.

> **Nota para no sobre-diseñar:** siete funciones para un análisis de finanzas personales está
> bien. Setenta estaría mal. La regla no es "cuantas más funciones mejor": es que cada paso que
> *podría fallar por separado* merece poder revisarse por separado. Si una función se usa una sola
> vez y es de una línea, probablemente no hacía falta.

<h2 style='font-weight:bold; color:#003366;'>6. `*args` y `**kwargs`: existen, y no los vamos a usar</h2>

Si leés código de librerías vas a encontrar firmas como esta:

```python
def resumir(*montos, **opciones):
    ...
```

- `*args` junta todos los argumentos **posicionales** extra en una tupla.
- `**kwargs` junta todos los argumentos **con nombre** extra en un diccionario.

Los nombres `args` y `kwargs` son pura convención; lo que importa son los asteriscos. Sirven para
escribir funciones que reciben una cantidad variable de cosas, y por eso aparecen tanto en el
código de `pandas` y `matplotlib`, que tienen que pasarle cuarenta opciones a otra función sin
declararlas una por una.

**En este curso no los vamos a usar.** Para nuestros análisis, una lista como parámetro y unos
pocos argumentos con valor por omisión son más claros y más fáciles de revisar: con `**kwargs` no
hay manera de saber, leyendo la firma, qué se le puede pasar a la función.

Con esto alcanza: reconocerlos cuando los vean en la documentación de una librería.

<h2 style='font-weight:bold; color:#003366;'>7. Caso aplicado: el módulo de finanzas personales</h2>

Acá es donde pasamos la mayor parte de la clase. Vamos a escribir, de a una función por vez, el
módulo que quedó planteado en la sección 5.

Los datos son `transacciones_personales.csv`, un extracto de dieciocho meses con los problemas
que tienen los datos reales: algunos montos ilegibles y muchas categorías vacías.

In [ ]:
import csv
from pathlib import Path


def ruta_datos(nombre_archivo):
    """Busca el archivo en las ubicaciones habituales del curso, incluida Colab."""
    candidatas = [
        Path("../Datos") / nombre_archivo,
        Path("Datos") / nombre_archivo,
        Path("/content/Datos") / nombre_archivo,
        Path(nombre_archivo),
    ]
    for candidata in candidatas:
        if candidata.exists():
            return str(candidata)
    raise FileNotFoundError(f"No se encontró {nombre_archivo}.")


def cargar_transacciones(ruta_archivo):
    """Lee el CSV y devuelve las transacciones válidas y cuántas filas se descartaron.
    Se descarta una fila cuando el monto no se puede convertir a número.
    """
    transacciones = []
    descartadas = 0

    with open(ruta_archivo, encoding="utf-8") as archivo:
        for fila in csv.DictReader(archivo):
            try:
                fila["monto"] = float(fila["monto"])
            except (ValueError, TypeError):
                descartadas += 1
                continue
            transacciones.append(fila)

    return transacciones, descartadas


transacciones, filas_descartadas = cargar_transacciones(
    ruta_datos("transacciones_personales.csv")
)

print(f"Transacciones válidas : {len(transacciones)}")
print(f"Filas descartadas     : {filas_descartadas}")
print(f"Período               : {transacciones[0]['fecha']} a {transacciones[-1]['fecha']}")
print(f"Columnas              : {list(transacciones[0].keys())}")

Transacciones válidas : 348
Filas descartadas     : 6
Período               : 2025-01-01 a 2026-06-28
Columnas              : ['id', 'fecha', 'descripcion', 'monto', 'categoria', 'medio_pago']


**Interpretación.** Entraron 348 transacciones y se descartaron 6, o sea el 1,7% de las
filas. Es una proporción baja y no cambia ninguna conclusión, pero el dato tiene que estar a la
vista: si mañana ese número fuera 80, el análisis no se puede publicar sin explicar por qué.

Fijate que `cargar_transacciones` **devuelve dos cosas**: los datos y el diagnóstico. Eso es lo
que la hace revisable.

In [19]:
# Cuántas categorías vienen vacías, que es el problema real de este archivo
sin_categoria = [t for t in transacciones if not t["categoria"]]

print(f"Transacciones sin categoría: {len(sin_categoria)} "
      f"({len(sin_categoria) / len(transacciones):.1%} del total)")
print("\nDescripciones que hay que imputar:")
for descripcion in sorted({t["descripcion"] for t in sin_categoria}):
    print("  -", descripcion)


SIN_CLASIFICAR = "Sin clasificar"


def clasificar_por_palabra_clave(descripcion, reglas):
    """Devuelve la primera categoría cuya palabra clave aparece en la descripción.

    El orden de las reglas importa: gana la primera que coincide.
    """
    texto = descripcion.lower()
    for categoria, palabras in reglas.items():
        if any(palabra in texto for palabra in palabras):
            return categoria
    return SIN_CLASIFICAR

Transacciones sin categoría: 29 (8.3% del total)

Descripciones que hay que imputar:
  - Antel hogar
  - Compra de UI
  - Cuota mutualista
  - Cuota universidad
  - Curso online
  - Disco
  - Fondo de inversion
  - Gastos comunes
  - Libros
  - Netflix
  - OSE
  - Odontologo
  - Peaje
  - Restaurante
  - Service auto
  - Spotify
  - Teatro Solis
  - Tienda Inglesa
  - Transferencia caja de ahorro
  - UTE
  - Uber


> La celda que sigue imputa las categorías vacías buscando palabras clave en la descripción.
> `PALABRAS_OCIO` está **incompleta a propósito**.
>
> **Corran la celda, miren la lista de descripciones que quedaron sin clasificar, y dicten las
> palabras que faltan.** 
>
> - ¿Cuántas transacciones quedan sin clasificar ahora?
> - ¿Alcanza con una palabra por servicio, o hay una más general que resuelve varias?

In [28]:
PALABRAS_OCIO = ["cine", "restaurante"]   # agreguen acá lo que les parezca que corresponde a ocio

REGLAS_CATEGORIA = {
    "Supermercado": ["tienda inglesa", "devoto", "disco", "ta-ta", "macromercado"],
    "Vivienda": ["alquiler", "gastos comunes", "ute", "ose", "antel"],
    "Transporte": ["combustible", "ancap", "stm", "uber", "peaje", "service"],
    "Salud": ["farmacia", "mutualista", "odontologo", "optica"],
    "Educacion": ["universidad", "libros", "curso"],
    "Ingresos": ["salario", "aguinaldo", "freelance", "reintegro"],
    "Ahorro e inversion": ["caja de ahorro", "compra de ui", "fondo de inversion"],
    "Ocio": PALABRAS_OCIO,
}

# Probamos las reglas SIN modificar los datos todavía: solo miramos qué pasaría
for t in sin_categoria:
    t["categoria"] = clasificar_por_palabra_clave(t["descripcion"], REGLAS_CATEGORIA)
    if t["categoria"] == SIN_CLASIFICAR:
        print(f"Sin clasificar: {t['descripcion']}")


Sin clasificar: Netflix
Sin clasificar: Teatro Solis
Sin clasificar: Spotify


In [30]:
transacciones

[{'id': '1',
  'fecha': '2025-01-01',
  'descripcion': 'Cuota universidad',
  'monto': -1197.39,
  'categoria': 'Educacion',
  'medio_pago': 'Debito'},
 {'id': '2',
  'fecha': '2025-01-03',
  'descripcion': 'Gastos comunes',
  'monto': -4617.31,
  'categoria': 'Vivienda',
  'medio_pago': 'Credito'},
 {'id': '3',
  'fecha': '2025-01-05',
  'descripcion': 'Salario mensual',
  'monto': 50572.38,
  'categoria': 'Ingresos',
  'medio_pago': 'Transferencia'},
 {'id': '4',
  'fecha': '2025-01-08',
  'descripcion': 'Cuota universidad',
  'monto': -1730.33,
  'categoria': 'Educacion',
  'medio_pago': 'Efectivo'},
 {'id': '5',
  'fecha': '2025-01-10',
  'descripcion': 'Fondo de inversion',
  'monto': -1771.94,
  'categoria': 'Ahorro e inversion',
  'medio_pago': 'Credito'},
 {'id': '6',
  'fecha': '2025-01-10',
  'descripcion': 'Spotify',
  'monto': -541.67,
  'categoria': 'Ocio',
  'medio_pago': 'Transferencia'},
 {'id': '7',
  'fecha': '2025-01-13',
  'descripcion': 'Fondo de inversion',
  'mon

In [31]:
def completar_categorias(transacciones, reglas):
    """Rellena las categorías vacías y devuelve cuántas quedaron sin clasificar."""
    sin_clasificar = 0
    for transaccion in transacciones:
        if not transaccion["categoria"]:
            transaccion["categoria"] = clasificar_por_palabra_clave(transaccion["descripcion"], reglas)
            if transaccion["categoria"] == SIN_CLASIFICAR:
                sin_clasificar += 1
    return sin_clasificar

pendientes_finales = completar_categorias(transacciones, REGLAS_CATEGORIA)
print(f"Transacciones sin clasificar después de imputar: {pendientes_finales}")

categorias = []
for t in transacciones:
    if t["categoria"] not in categorias:
        categorias.append(t["categoria"])
        
print("Categorías finales:", categorias)
print()

# Con los datos ya limpios y clasificados, recién ahora se puede calcular
def separar_ingresos_y_egresos(transacciones):
    ingresos = []
    egresos = []
    for t in transacciones:
        if t["monto"] > 0:
            ingresos.append(t)
        elif t["monto"] < 0:
            egresos.append(t)
    return ingresos, egresos


def sumar_montos(transacciones):
    monto_total = 0
    for transaccion in transacciones:
        monto_total += transaccion["monto"]
    return monto_total


def calcular_tasa_ahorro(total_ingresos, total_egresos):
    """Proporción de los ingresos que no se gastó. Los egresos llegan en negativo.

    No descuenta las transferencias a cuentas de ahorro, que en estos datos figuran
    como egreso.
    """
    if total_ingresos == 0:
        raise ValueError("No se puede calcular la tasa de ahorro sin ingresos")
    return (total_ingresos + total_egresos) / total_ingresos


ingresos, egresos = separar_ingresos_y_egresos(transacciones)
total_ingresos = sumar_montos(ingresos)
total_egresos = sumar_montos(egresos)

print(f"Ingresos del período : $U {total_ingresos:>12,.0f}  ({len(ingresos)} movimientos)")
print(f"Egresos del período  : $U {abs(total_egresos):>12,.0f}  ({len(egresos)} movimientos)")
print(f"Balance              : $U {total_ingresos + total_egresos:>12,.0f}")
print(f"Tasa de ahorro       : {calcular_tasa_ahorro(total_ingresos, total_egresos):>13.1%}")

Transacciones sin clasificar después de imputar: 0
Categorías finales: ['Educacion', 'Vivienda', 'Ingresos', 'Ahorro e inversion', 'Ocio', 'Salud', 'Supermercado', 'Transporte', 'Sin clasificar']

Ingresos del período : $U    1,006,857  (24 movimientos)
Egresos del período  : $U      695,647  (324 movimientos)
Balance              : $U      311,210
Tasa de ahorro       :         30.9%


**Interpretación económica.** Dieciocho meses con $U 1.006.857 de ingresos y
$U 695.647 de egresos dejan un excedente de $U 311.210 y una **tasa de ahorro** del 30,9%.

Dos lecturas que hay que hacer antes de festejar:

1. **La tasa de ahorro está subestimada.** Dentro de los egresos hay $U 151.590 de la categoría
   "Ahorro e inversión", que son transferencias a caja de ahorro y compras de UI. Eso no es
   consumo: es ahorro que ya se hizo. Contando bien, la proporción de ingresos que no se consumió
   es del 46%. Esto está aclarado en el docstring de `calcular_tasa_ahorro`, y es exactamente el
   tipo de supuesto que justifica escribir un docstring de dos líneas en lugar de ninguna.
2. **24 movimientos de ingreso contra 324 de egreso.** El ingreso está concentrado en el salario:
   si se corta esa única fuente, el presupuesto no tiene nada que lo sostenga. Es el equivalente
   doméstico de la concentración de clientes en una empresa.

In [ ]:
from collections import defaultdict


def resumir_por_mes(transacciones):
    """Devuelve un diccionario con: mes ingresos, egresos y balance.

    El mes se toma de los primeros siete caracteres de la fecha, o sea 'AAAA-MM'.
    """
    resumen = defaultdict(lambda: {"ingresos": 0.0, "egresos": 0.0})
    for transaccion in transacciones:
        mes = transaccion["fecha"][:7]
        if transaccion["monto"] > 0:
            resumen[mes]["ingresos"] += transaccion["monto"]
        else:
            resumen[mes]["egresos"] += transaccion["monto"]

    for datos in resumen.values():
        datos["balance"] = datos["ingresos"] + datos["egresos"]

    return dict(resumen)


evolucion = resumir_por_mes(transacciones)

print("Mes       Ingresos      Egresos      Balance   Tasa de ahorro")
for mes in sorted(evolucion):
    datos = evolucion[mes]
    tasa = calcular_tasa_ahorro(datos["ingresos"], datos["egresos"])
    print(f"{mes}  {datos['ingresos']:>10,.0f}  {abs(datos['egresos']):>11,.0f}  "
          f"{datos['balance']:>11,.0f}   {tasa:>8.1%}")

Mes       Ingresos      Egresos      Balance   Tasa de ahorro
2025-01      50,572       37,424       13,149      26.0%
2025-02      50,357       32,719       17,638      35.0%
2025-03      50,295       33,083       17,211      34.2%
2025-04      53,003       43,281        9,722      18.3%
2025-05      52,389       49,595        2,794       5.3%
2025-06      79,335       27,923       51,412      64.8%
2025-07      50,001       43,791        6,210      12.4%
2025-08      53,922       34,307       19,615      36.4%
2025-09      52,385       42,646        9,739      18.6%
2025-10      53,299       45,057        8,242      15.5%
2025-11      52,692       26,031       26,661      50.6%
2025-12      78,085       51,213       26,872      34.4%
2026-01      52,214       45,436        6,777      13.0%
2026-02      50,332       46,224        4,109       8.2%
2026-03      48,194       28,474       19,720      40.9%
2026-04      52,649       22,829       29,820      56.6%
2026-05      53,464       

> Con las funciones que acabamos de escribir, y sin escribir ninguna nueva:
>
> **¿Cuál fue el mes con peor tasa de ahorro, cuánto fue esa tasa, y qué explica que haya sido
> tan mala?**
>
> La clase dicta los pasos:
>
> 1. ¿Qué función nos da los datos por mes?
> 2. ¿Cómo elegimos el mínimo de un diccionario? 
> 3. Una vez que tenemos el mes, ¿cómo miramos qué se gastó ese mes?

In [ ]:
tasa_minima = 100
mes_peor_ahorro = ""
for mes in sorted(evolucion):
    datos = evolucion[mes]
    tasa = calcular_tasa_ahorro(datos["ingresos"], datos["egresos"])
    if tasa < tasa_minima:
        tasa_minima = tasa
        mes_peor_ahorro = mes

datos_del_mes = evolucion[mes_peor_ahorro]

print(f"Mes con peor tasa de ahorro: {mes_peor_ahorro}")
print(f"  Ingresos: $ {datos_del_mes['ingresos']:,.0f}")
print(f"  Egresos: $ {abs(datos_del_mes['egresos']):,.0f}")
print(f"  Balance: $ {datos_del_mes['balance']:,.0f}")
print(f"  Tasa de ahorro: {tasa_minima:.1%}")

# Y ahora miramos qué se gastó ese mes, por categoría
gastos_del_mes = {}
for transaccion in transacciones:
    if transaccion["fecha"][:7] == mes_peor_ahorro and transaccion["monto"] < 0:

        if transaccion["categoria"] not in gastos_del_mes:
            gastos_del_mes[transaccion["categoria"]] = 0

        gastos_del_mes[transaccion["categoria"]] += transaccion["monto"]

print(f"\nGasto de {mes_peor_ahorro} por categoría:")

for categoria, monto in gastos_del_mes.items():
    peso = abs(monto) / abs(datos_del_mes["egresos"])
    print(f"  {categoria} ${abs(monto):,.0f}   {peso:.1%}")

Mes con peor tasa de ahorro: 2025-05
  Ingresos: $ 52,389
  Egresos: $ 49,595
  Balance: $ 2,794
  Tasa de ahorro: 5.3%

Gasto de 2025-05 por categoría:
  Transporte $2,613   5.3%
  Supermercado $9,349   18.9%
  Educacion $5,526   11.1%
  Ocio $2,053   4.1%
  Vivienda $17,753   35.8%
  Ahorro e inversion $10,082   20.3%
  Salud $2,219   4.5%


**El número de la clase: mayo de 2025, con una tasa de ahorro del 5,3%.**

Ingresos de $U 52.389, egresos de $U 49.595, excedente de $U 2.794. Un mes en el que
prácticamente todo lo que entró se gastó.

Y lo que importa no es el número, es la explicación. Lo interesante del desglose es lo que **no**
muestra: no hay un gasto extraordinario que explique el mes. Frente a un promedio de $U 38.647 de
egresos mensuales, mayo gastó $U 10.948 más, pero repartidos: supermercado casi $U 4.000 arriba de
su promedio, vivienda $U 2.600, ahorro e inversión $U 1.700, educación $U 1.300. Ningún renglón se
descontroló; se descontrolaron todos un poco a la vez.

Ese es el diagnóstico más incómodo y también el más frecuente. Cuando el mes malo se explica por
un gasto único, la respuesta es un fondo de reserva. Cuando se explica por cuatro categorías que
subieron un 20% cada una, la respuesta es que el nivel de gasto está pegado al ingreso y no hay
colchón.

<h2 style='font-weight:bold; color:#003366; text-align:center;'>Ejercicios para resolver en clase</h2>

Los tres ejercicios trabajan sobre `transacciones`, que ya está cargado y con las categorías
completas. La regla es la misma para todos: **las funciones calculan y devuelven, los `print`
van afuera.**

1. Leé el enunciado y pensá qué devuelve la función antes de escribirla.
2. Resolvelo en la celda vacía que sigue.
3. Recién después desplegá la solución para comparar los enfoques.

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 1 *</h3>

Escribí una función `contar_por_categoria(transacciones)` que devuelva un
diccionario con **cuántas transacciones** hay en cada categoría.

Responder: ¿la categoría con más movimientos es la que más plata mueve?

<details>
<summary><b>Ver solución</b></summary>

```python
def contar_por_categoria(transacciones):
    conteo = {}
    for transaccion in transacciones:
        categoria = transaccion["categoria"]
        conteo[categoria] = conteo.get(categoria, 0) + 1
    return conteo


conteo = contar_por_categoria(transacciones)
for categoria, cantidad in conteo.items():
    print(f"{categoria:<22} {cantidad:>4} movimientos")

# Vivienda tiene 57 movimientos y Educación 55, casi lo mismo. Pero Vivienda gasta
# $U 272.645 y Educación $U 75.321: tres veces y media más con la misma cantidad de
# movimientos. 
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 2 *</h3>

Escribí una función `filtrar_por_medio_pago(transacciones, medio_pago)` que
devuelva solamente las transacciones pagadas con ese medio.

Usala para contestar: **¿cuántos movimientos se pagaron con tarjeta de crédito y cuánto suman?**
Podés usar `sumar_montos`, que ya está escrita.

In [46]:
# Tu solución acá
  

<details>
<summary><b>Ver solución</b></summary>

```python
def filtrar_por_medio_pago(transacciones, medio_pago):
    lista_transacciones = []
    for t in transacciones:
        if t["medio_pago"] == medio_pago:
            lista_transacciones.append(t) 
    return lista_transacciones    

con_credito = filtrar_por_medio_pago(transacciones, "Credito")

print(f"Movimientos con crédito: {len(con_credito)}")
print(f"Total movido           : $U {abs(sumar_montos(con_credito)):,.0f}")

# 83 movimientos por $U 167.396 de gasto. Es el 24,1% del gasto total del período: casi
# una cuarta parte del consumo pasa por un instrumento que corre el pago un mes. Eso no
# es malo por sí mismo, pero significa que el saldo de un mes cualquiera no refleja lo
# que realmente se gastó ese mes.
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 3 **</h3>

**Combiná tres funciones.** Escribí `gasto_por_medio_de_pago(transacciones)`
que devuelva un diccionario con el **gasto total** (solo egresos) de cada medio de pago, usando `separar_ingresos_y_egresos` para obtener los egresos y luego
`filtrar_por_medio_pago` y `sumar_montos`.

Devolvé los montos en positivo, que es como se leen en un informe. Después interpretá: ¿está
concentrado en un medio o repartido?


Dos pasos:

- Primero necesitás la lista de medios de pago que existen en los egresos, sin repetir.
- Después, para cada medio, filtrá y sumá. Ojo con quedarte solo con los egresos: `filtrar_egresos`
  no existe, pero `separar_ingresos_y_egresos` sí, y devuelve dos listas.


In [50]:
# Tu solución acá


<details>
<summary><b>Ver solución</b></summary>

```python
def gasto_por_medio_de_pago(transacciones):
    """Gasto total de cada medio de pago, en positivo. Ignora los ingresos."""
    ingresos, egresos = separar_ingresos_y_egresos(transacciones)
    medios_de_pago = []
    for t in egresos:
        # Valores unicos de la clave "medio_pago" de los egresos
        if t["medio_pago"] not in medios_de_pago:
            medios_de_pago.append(t["medio_pago"])

    gasto_por_medio = {}
    for medio in medios_de_pago:
        gasto_por_medio[medio] = abs(sumar_montos(filtrar_por_medio_pago(egresos, medio)))
    return gasto_por_medio

gasto_por_medio = gasto_por_medio_de_pago(transacciones)
gasto_total = sum(gasto_por_medio.values())

for medio, monto in sorted(gasto_por_medio.items(), key=lambda par: par[1], reverse=True):
    print(f"{medio:<15} $U {monto:>10,.0f}   {monto / gasto_total:>5.1%}")
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 4 ***</h3>

**Agregá una función más al módulo.** Escribí
`calcular_ticket_promedio(transacciones, categoria)`, que devuelva el gasto promedio por
transacción en esa categoría. El **ticket promedio** es el gasto total dividido la cantidad de
operaciones, y es uno de los indicadores más usados en análisis comercial.

Requisitos:

1. Si la categoria no existe comunicar con un print y no devolver 0.
2. Si la categoría existe pero no tiene egresos, devolvé 0 en lugar de dividir por cero.
3. Usala para calcular el ticket promedio de todas las categorías de gasto y contestá: **¿cuál
   tiene el ticket más alto y qué significa eso en términos de manejo del presupuesto?**

<details>
<summary><b>Ver solución</b></summary>

```python
def calcular_ticket_promedio(transacciones, categoria):
    """Gasto promedio por operación en una categoría, en positivo.
    Solo considera egresos: un reintegro no baja el ticket promedio de la categoría.
    """
    categorias_existentes = []
    for t in transacciones:
        if t["categoria"] not in categorias_existentes:
            categorias_existentes.append(t["categoria"])

    if categoria not in categorias_existentes:
        print("La categoria que quiere no existe en los datos. Las disponibles son:")
        print("Categorías existentes:", categorias_existentes)
        return 0

    _, egresos = separar_ingresos_y_egresos(transacciones)
    egresos_de_la_categoria = []
    for transaccion in egresos:
        if transaccion["categoria"] == categoria:
            egresos_de_la_categoria.append(transaccion)
    if egresos_de_la_categoria == []:
        return 0.0

    egreso_promedio = abs(sumar_montos(egresos_de_la_categoria)) / len(egresos_de_la_categoria)

    return egreso_promedio



calcular_ticket_promedio(transacciones, "Viajes")

for categoria in ['Educacion', 'Vivienda', 'Ingresos', 'Ahorro e inversion', 'Ocio', 'Salud', 'Supermercado', 'Transporte', 'Sin clasificar']:
    print(f"{categoria}: {calcular_ticket_promedio(transacciones, categoria):,.0f}")

```

</details>